In [1]:
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

#Task 1

In [2]:
outputPath1="./output1/"
meetingsRDD=sc.textFile("./data/meetings.txt")

In [3]:
cleanedMeetingRDD=meetingsRDD.map(lambda x: (x.split(",")[4],(int(x.split(",")[3]),1))) \
    .reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1])).map(lambda x: (x[0],x[1][0]/x[1][1]))

maxDur=cleanedMeetingRDD.values().max()

finalRDD=cleanedMeetingRDD.filter(lambda x: x[1]==maxDur).reduce(lambda a, b: a if a[0] < b[0] else b)

#Task 2

In [7]:
outputPath2="./output2/"
invitationsRDD=sc.textFile("./data/invitations.txt")
participationsRDD=sc.textFile("./data/participations.txt")

In [54]:
def mappaggio(row):
  meetId=row.split(",")[0]
  meetDate=row.split(",")[2]
  if meetDate<"2023/01/01-00:00":
    return (meetId,1)
  else:
    return (meetId,0)

newCleanedMeetingRDD=meetingsRDD.map(mappaggio)
cleanedParticipationsRDD=participationsRDD.map(lambda x: ((x.split(",")[0],x.split(",")[1]),(0,1))).distinct()

cleanedInvitationsRDD=invitationsRDD.map(lambda x: (x.split(",")[0],(x.split(",")[1],1 if x.split(",")[2]=='Yes' else 0)))

joinedRDD=cleanedInvitationsRDD.join(newCleanedMeetingRDD).filter(lambda x: x[1][1]==1) \
    .map(lambda x: ((x[0],x[1][0][0]),(x[1][0][1],0)))

mismatchRDD=joinedRDD.union(cleanedParticipationsRDD).reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1])) \
    .filter(lambda x: x[1][0]!=x[1][1]).map(lambda x: (x[0][1],x[1])).reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1]))



In [55]:
mismatchRDD.collect()

[('Cust3', (2, 1)), ('Cust4', (3, 0)), ('Cust2', (0, 1))]